In [2]:
import polars as pl
import pyprojroot

ROOT = pyprojroot.here()

In [4]:
datos = pl.read_parquet(ROOT/"data"/"raw"/"temporada1.parquet")

Saqué un par de variables de acá https://www.kaggle.com/code/thiagomantuani/predict-hits-with-new-mlb-get-started-here/notebook

In [ ]:
# Constante de conversión: 1 pie = 30.48 cm
FT_TO_CM = 30.48

datos_nuevos = (
    datos.with_columns(
        plate_x = (pl.col('plate_x') * FT_TO_CM),
        plate_z = (pl.col('plate_z') * FT_TO_CM),
        pfx_x = (pl.col('pfx_x') * FT_TO_CM),
        pfx_z = (pl.col('pfx_z') * FT_TO_CM),
        sz_top = (pl.col('sz_top') * FT_TO_CM),
        sz_bot = (pl.col('sz_bot') * FT_TO_CM)
    )
    .with_columns(
        pitch_location = (pl.col('plate_x')**2 + pl.col('plate_z')**2).sqrt(),
        movement_complexity = (pl.col('pfx_x')**2 + pl.col('pfx_z')**2).sqrt(),
        count_pressure = pl.col('balls') - pl.col('strikes') + 1,
        platoon_advantage = (pl.col('p_throws') != pl.col('stand')).cast(pl.Int32),
        strike_zone_size = pl.col('sz_top') - pl.col('sz_bot'),
    )
    .with_columns(
        relative_height = (pl.col('plate_z') - pl.col('sz_bot')) / (pl.col('sz_top') - pl.col('sz_bot')),
    )
    .with_columns(
        # 8.5 pulgadas = 21.589 cm. Esto representa la mitad de las 17 pulgadas de ancho de la zona de strike.
        is_strike_zone = (pl.col('plate_x').is_between(-21.59, 21.59) & pl.col('relative_height').is_between(0, 1)).cast(pl.Int32),
    )
    .with_columns(
        # 1. Cuenta pitcheos alejados del centro SOLO si cayeron en la zona de strike
        pitch_location_effectiveness = pl.col('pitch_location') * pl.col('is_strike_zone'),
        
        # 2. Situaciones críticas
        is_two_strikes = (pl.col('strikes') == 2).cast(pl.Int32),
        is_full_count = ((pl.col('balls') == 3) & (pl.col('strikes') == 2)).cast(pl.Int32)

        # Multiplica la velocidad (usualmente en mph) por el movimiento total en cm
        pitch_stuff_proxy = pl.col('release_speed') * pl.col('movement_complexity'),
    )
)